# PFA N°13 — Sprint 3 : Tests du moteur de règles expertes

Ce notebook regroupe tous les tests réalisés durant le Sprint 3 :

1. Évaluateur de conditions (`condition_evaluator.py`)
2. Moteur d'inférence et logique règle mère / règle fille (`rule_engine.py`)
3. Chargement et vérification du corpus réel (53 règles / 53 mesures)
4. Calcul du score de priorité (`scoring.py`)
5. Test complet du pipeline sur 6 profils PME fictifs contrastés
6. Exécution de la suite de tests automatisés (`pytest`)

Objectif : disposer d'un support unique, exécutable et commentable, pour le
rapport de test attendu par Pr. Ajana à l'issue du Sprint 3.


## 0. Setup

In [1]:
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd

# backend/ est le dossier parent de notebooks/
BACKEND_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BACKEND_DIR))

from app.engine.condition_evaluator import evaluate_condition, ConditionEvaluationError
from app.engine.rule_engine import RuleEngine
from app.engine.scoring import multiplicateur_contexte, pertinence_secteur, calculer_score_priorite
from app.engine.recommender import generer_plan_action

print(f"BACKEND_DIR = {BACKEND_DIR}")


BACKEND_DIR = c:\Users\hp\cyber-pme\backend


In [2]:
def trouver_dossier_seeds():
    """Cherche database/seeds/ dans les emplacements plausibles du repo."""
    candidats = [
        BACKEND_DIR.parent / "database" / "seeds",  # cyber-pme/database/seeds/
        BACKEND_DIR / "database" / "seeds",           # variante si database/ est sous backend/
        Path("/mnt/project"),                         # environnement de dev Claude
    ]
    for c in candidats:
        if (c / "mesures.json").exists():
            return c
    raise FileNotFoundError(
        "mesures.json / regles_expertes.json introuvables. "
        "Placez-les dans database/seeds/ à la racine du repo (sibling de backend/)."
    )

SEEDS_DIR = trouver_dossier_seeds()
print(f"Fichiers seeds trouvés dans : {SEEDS_DIR}")

with open(SEEDS_DIR / "mesures.json", encoding="utf-8") as f:
    mesures = json.load(f)
with open(SEEDS_DIR / "regles_expertes.json", encoding="utf-8") as f:
    regles = json.load(f)

print(f"{len(mesures)} mesures chargées, {len(regles)} règles chargées.")


Fichiers seeds trouvés dans : c:\Users\hp\cyber-pme\database\seeds
53 mesures chargées, 53 règles chargées.


## 1. Évaluateur de conditions

Vérifie les trois formes de conditions utilisées dans `regles_expertes.json` :
condition simple, condition composée avec clause `et` (égalité), et clause
`et` avec `contient` (appartenance à un tableau, ex. `reglementations_applicables`).


In [3]:
exemples_conditions = [
    ("Simple, satisfaite", {"id_question": 10, "operateur": "<=", "valeur": 1}, {10: 1}, {}),
    ("Simple, non satisfaite", {"id_question": 10, "operateur": "<=", "valeur": 1}, {10: 3}, {}),
    ("Question sans réponse -> False (pas d'erreur)", {"id_question": 10, "operateur": "<=", "valeur": 1}, {}, {}),
    (
        "Composée 'et' valeur, satisfaite",
        {"id_question": 17, "operateur": "<=", "valeur": 1, "et": {"champ": "traite_donnees_sensibles", "valeur": True}},
        {17: 1},
        {"traite_donnees_sensibles": True},
    ),
    (
        "Composée 'et' valeur, clause fausse",
        {"id_question": 17, "operateur": "<=", "valeur": 1, "et": {"champ": "traite_donnees_sensibles", "valeur": True}},
        {17: 1},
        {"traite_donnees_sensibles": False},
    ),
    (
        "Composée 'et' contient (RGPD), satisfaite",
        {"id_question": 24, "operateur": "<=", "valeur": 1, "et": {"champ": "reglementations_applicables", "contient": "rgpd"}},
        {24: 1},
        {"reglementations_applicables": ["loi_09_08", "rgpd"]},
    ),
]

lignes = []
for libelle, condition, reponses, profil in exemples_conditions:
    resultat = evaluate_condition(condition, reponses, profil)
    lignes.append({"Cas": libelle, "Résultat": resultat})

pd.DataFrame(lignes)


,Cas,Résultat
0,"Simple, satisfaite",True
1,"Simple, non satisfaite",False
2,Question sans réponse -> False (pas d'erreur),False
3,"Composée 'et' valeur, satisfaite",True
4,"Composée 'et' valeur, clause fausse",False
5,"Composée 'et' contient (RGPD), satisfaite",True


In [4]:
# Cas d'erreur volontaire : opérateur inconnu
try:
    evaluate_condition({"id_question": 10, "operateur": "~=", "valeur": 1}, {10: 1}, {})
except ConditionEvaluationError as e:
    print(f"Erreur correctement levée : {e}")


Erreur correctement levée : Opérateur inconnu : ~=


## 2. Moteur d'inférence — logique règle mère / règle fille

Corpus synthétique minimal pour isoler la logique de propagation
mère → fille → petite-fille, indépendamment des vraies données.


In [5]:
mesures_synth = [
    {"id_mesure": 1, "titre": "Mesure A (mère)", "impact": "moyen"},
    {"id_mesure": 2, "titre": "Mesure B (fille)", "impact": "tres_eleve"},
    {"id_mesure": 3, "titre": "Mesure C (petite-fille)", "impact": "eleve"},
]

regles_synth = [
    {
        "id_regle": 1, "id_domaine": 1, "id_regle_parent": None,
        "condition": {"id_question": 10, "operateur": "<=", "valeur": 1},
        "id_mesure": 1, "priorite_base": 3,
    },
    {
        "id_regle": 2, "id_domaine": 1, "id_regle_parent": 1,
        "condition": {"id_question": 10, "operateur": "<=", "valeur": 1,
                      "et": {"champ": "traite_donnees_sensibles", "valeur": True}},
        "id_mesure": 2, "priorite_base": 5,
    },
    {
        "id_regle": 3, "id_domaine": 1, "id_regle_parent": 2,
        "condition": {"id_question": 10, "operateur": "<=", "valeur": 1,
                      "et": {"champ": "budget_cybersecurite", "valeur": "aucun"}},
        "id_mesure": 3, "priorite_base": 5,
    },
]

engine_synth = RuleEngine(regles_synth, mesures_synth)

cas_test = [
    ("Mère seule (pas de données sensibles)", {10: 1}, {"traite_donnees_sensibles": False, "budget_cybersecurite": "faible"}),
    ("Mère + fille (données sensibles, budget non nul)", {10: 1}, {"traite_donnees_sensibles": True, "budget_cybersecurite": "modere"}),
    ("Chaîne complète (données sensibles + budget aucun)", {10: 1}, {"traite_donnees_sensibles": True, "budget_cybersecurite": "aucun"}),
    ("Parent non déclenché -> rien, même si clauses filles seraient vraies", {10: 3}, {"traite_donnees_sensibles": True, "budget_cybersecurite": "aucun"}),
]

for libelle, reponses, profil in cas_test:
    resultats = engine_synth.evaluer(reponses, profil)
    ids = sorted(r.id_regle for r in resultats)
    print(f"{libelle}\n  -> règles déclenchées : {ids}\n")


Mère seule (pas de données sensibles)
  -> règles déclenchées : [1]

Mère + fille (données sensibles, budget non nul)
  -> règles déclenchées : [1, 2]

Chaîne complète (données sensibles + budget aucun)
  -> règles déclenchées : [1, 2, 3]

Parent non déclenché -> rien, même si clauses filles seraient vraies
  -> règles déclenchées : []



## 3. Vérification du corpus réel (53 règles / 53 mesures)

Contrôle que les 3 vrais couples mère/fille du guide CMRPI/AUSIM
(règles 3/4, 22/23, 31/32 — toutes conditionnées par
`traite_donnees_sensibles=true`) se comportent comme attendu.


In [6]:
engine = RuleEngine(regles, mesures)

assert len(regles) == 53
assert len(mesures) == 53

profil_sensible = {"traite_donnees_sensibles": True}
profil_non_sensible = {"traite_donnees_sensibles": False}
reponses_test = {17: 1, 21: 1, 23: 1}  # déclenche les 3 règles mères (3, 22, 31)

ids_sensible = {r.id_regle for r in engine.evaluer(reponses_test, profil_sensible)}
ids_non_sensible = {r.id_regle for r in engine.evaluer(reponses_test, profil_non_sensible)}

lignes = []
for fille, mere in [(4, 3), (23, 22), (32, 31)]:
    lignes.append({
        "Règle mère": mere,
        "Règle fille": fille,
        "Déclenchée si données sensibles": fille in ids_sensible,
        "Déclenchée si PAS de données sensibles": fille in ids_non_sensible,
    })

pd.DataFrame(lignes)


,Règle mère,Règle fille,Déclenchée si données sensibles,Déclenchée si PAS de données sensibles
0,3,4,True,False
1,22,23,True,False
2,31,32,True,False


## 4. Scoring — formule `score_priorite`

    score_priorite = priorite_base x poids_impact x multiplicateur_contexte x pertinence_secteur

Vérification des bornes et du comportement attendu de chaque facteur.


In [7]:
lignes = []

# multiplicateur_contexte : de 1.0 (aucun risque) à 2.0 (risque maximal)
lignes.append({"Facteur": "multiplicateur_contexte", "Cas": "aucune réponse Q1-Q9",
                "Valeur": multiplicateur_contexte({})})
lignes.append({"Facteur": "multiplicateur_contexte", "Cas": "toutes réponses = 0",
                "Valeur": multiplicateur_contexte({i: 0 for i in range(1, 10)})})
lignes.append({"Facteur": "multiplicateur_contexte", "Cas": "toutes réponses = 3 (risque max)",
                "Valeur": multiplicateur_contexte({i: 3 for i in range(1, 10)})})

# pertinence_secteur : bonus x1.2 si secteur sensible/données sensibles + impact fort
lignes.append({"Facteur": "pertinence_secteur", "Cas": "santé + impact tres_eleve",
                "Valeur": pertinence_secteur({"impact": "tres_eleve"}, {"secteur_activite": "sante", "traite_donnees_sensibles": False})})
lignes.append({"Facteur": "pertinence_secteur", "Cas": "commerce + impact tres_eleve (neutre)",
                "Valeur": pertinence_secteur({"impact": "tres_eleve"}, {"secteur_activite": "commerce", "traite_donnees_sensibles": False})})
lignes.append({"Facteur": "pertinence_secteur", "Cas": "impact faible même en secteur sensible (neutre)",
                "Valeur": pertinence_secteur({"impact": "faible"}, {"secteur_activite": "sante", "traite_donnees_sensibles": True})})

pd.DataFrame(lignes)


,Facteur,Cas,Valeur
0,multiplicateur_contexte,aucune réponse Q1-Q9,1.0
1,multiplicateur_contexte,toutes réponses = 0,1.0
2,multiplicateur_contexte,toutes réponses = 3 (risque max),2.0
3,pertinence_secteur,santé + impact tres_eleve,1.2
4,pertinence_secteur,commerce + impact tres_eleve (neutre),1.0
5,pertinence_secteur,impact faible même en secteur sensible (neutre),1.0


In [8]:
# Formule complète : cas neutre vs cas maximal
score_neutre = calculer_score_priorite(
    priorite_base=3, mesure={"impact": "moyen"}, reponses={},
    profil={"secteur_activite": "commerce", "traite_donnees_sensibles": False},
)
score_maximal = calculer_score_priorite(
    priorite_base=5, mesure={"impact": "tres_eleve"}, reponses={i: 3 for i in range(1, 10)},
    profil={"secteur_activite": "sante", "traite_donnees_sensibles": True},
)

print(f"Score cas neutre  (prio=3, impact=moyen, contexte neutre, secteur neutre)  = {score_neutre}")
print(f"Score cas maximal (prio=5, impact=tres_eleve, contexte max, secteur sensible) = {score_maximal}")


Score cas neutre  (prio=3, impact=moyen, contexte neutre, secteur neutre)  = 4.5
Score cas maximal (prio=5, impact=tres_eleve, contexte max, secteur sensible) = 30.0


## 5. Test complet sur 6 profils PME fictifs contrastés

Profils volontairement variés (secteur, taille, budget, données sensibles,
RGPD, historique d'incident, "forme" de maturité) pour éprouver le moteur
sur des situations réalistes différentes.


In [9]:
PROFILS = {
    "Clinique Santé Nord": (
        {"secteur_activite": "sante", "taille_effectif": "petite", "budget_cybersecurite": "aucun",
         "traite_donnees_sensibles": True, "reglementations_applicables": ["loi_09_08"]},
        {1: 2, 2: 3, 3: 2, 4: 3, 5: 1, 6: 0, 7: 2, 8: 1, 9: 2,
         10: 1, 11: 0, 12: 1, 13: 1, 14: 0, 15: 1, 16: 1, 17: 1,
         18: 2, 19: 1, 20: 1, 21: 1, 22: 0, 23: 1, 24: 1},
    ),
    "Cabinet Compta Rif": (
        {"secteur_activite": "services", "taille_effectif": "tpe", "budget_cybersecurite": "faible",
         "traite_donnees_sensibles": False, "reglementations_applicables": ["loi_09_08"]},
        {1: 1, 2: 1, 3: 1, 4: 1, 5: 0, 6: 0, 7: 1, 8: 0, 9: 0,
         10: 3, 11: 2, 12: 2, 13: 3, 14: 2, 15: 2, 16: 1, 17: 3,
         18: 3, 19: 3, 20: 3, 21: 3, 22: 2, 23: 2, 24: 2},
    ),
    "Usine Textile Atlas": (
        {"secteur_activite": "industrie", "taille_effectif": "moyenne", "budget_cybersecurite": "modere",
         "traite_donnees_sensibles": False, "reglementations_applicables": []},
        {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 0, 7: 1, 8: 1, 9: 1,
         10: 2, 11: 2, 12: 2, 13: 2, 14: 1, 15: 2, 16: 2, 17: 4,
         18: 2, 19: 1, 20: 2, 21: 0, 22: 1, 23: 2, 24: 3},
    ),
    "Startup FinTech Wave": (
        {"secteur_activite": "tech_digital", "taille_effectif": "tpe", "budget_cybersecurite": "structure",
         "traite_donnees_sensibles": True, "reglementations_applicables": ["loi_09_08", "rgpd"]},
        {1: 3, 2: 3, 3: 2, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 1,
         10: 3, 11: 3, 12: 2, 13: 3, 14: 2, 15: 3, 16: 3, 17: 2,
         18: 3, 19: 3, 20: 3, 21: 1, 22: 1, 23: 3, 24: 1},
    ),
    "Boutique Multimode": (
        {"secteur_activite": "commerce", "taille_effectif": "petite", "budget_cybersecurite": "aucun",
         "traite_donnees_sensibles": False, "reglementations_applicables": []},
        {1: 0, 2: 1, 3: 1, 4: 1, 5: 0, 6: 0, 7: 0, 8: 0, 9: 1,
         10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0, 16: 0, 17: 0,
         18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0},
    ),
    "Assurance Maghreb": (
        {"secteur_activite": "finance_assurance", "taille_effectif": "moyenne", "budget_cybersecurite": "structure",
         "traite_donnees_sensibles": True, "reglementations_applicables": ["loi_09_08", "rgpd"]},
        {1: 3, 2: 3, 3: 3, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 1,
         10: 4, 11: 4, 12: 3, 13: 4, 14: 3, 15: 4, 16: 1, 17: 5,
         18: 4, 19: 4, 20: 4, 21: 1, 22: 4, 23: 4, 24: 1},
    ),
}

plans = {}
for nom, (profil, reponses) in PROFILS.items():
    plans[nom] = generer_plan_action(engine, reponses, profil)

recap = pd.DataFrame([
    {
        "Entreprise": nom,
        "Secteur": PROFILS[nom][0]["secteur_activite"],
        "Nb recommandations": len(plan),
        "Score max": plan[0]["score_priorite"] if plan else 0.0,
        "Score moyen": round(sum(r["score_priorite"] for r in plan) / len(plan), 2) if plan else 0.0,
    }
    for nom, plan in plans.items()
])
recap


,Entreprise,Secteur,Nb recommandations,Score max,Score moyen
0,Clinique Santé Nord,sante,45,23.89,10.87
1,Cabinet Compta Rif,services,6,9.48,6.42
2,Usine Textile Atlas,industrie,14,16.20,9.53
3,Startup FinTech Wave,tech_digital,13,27.22,15.11
4,Boutique Multimode,commerce,49,14.35,6.56
5,Assurance Maghreb,finance_assurance,14,26.67,12.31


In [10]:
# Détail : top 5 recommandations pour chaque profil
for nom, plan in plans.items():
    print(f"\n=== {nom} ({len(plan)} recommandations) ===")
    top5 = pd.DataFrame(plan[:5])[["score_priorite", "id_domaine", "id_regle", "titre_mesure"]]
    display(top5)



=== Clinique Santé Nord (45 recommandations) ===


,score_priorite,id_domaine,id_regle,titre_mesure
0,23.89,8,4,Renforcer le verrouillage physique des postes ...
1,23.89,10,17,Restreindre les comptes administrateurs aux se...
2,23.89,12,22,Chiffrer les données sensibles avant tout stoc...
3,23.89,12,23,Mettre en place un certificat SSL (HTTPS) pour...
4,23.89,13,26,Définir une procédure formelle de réaction en ...



=== Cabinet Compta Rif (6 recommandations) ===


,score_priorite,id_domaine,id_regle,titre_mesure
0,9.48,7,45,Mettre en place un programme de sensibilisatio...
1,9.48,7,47,Sensibiliser les employés à ne jamais divulgue...
2,9.48,7,48,Former les employés à signaler au helpdesk (pl...
3,5.33,7,46,Former les employés à choisir un mot de passe ...
4,2.37,7,49,Former les employés au maintien d'un bureau cl...



=== Usine Textile Atlas (14 recommandations) ===


...

,score_priorite,id_domaine,id_regle,titre_mesure
0,27.22,12,22,Chiffrer les données sensibles avant tout stoc...
1,27.22,12,23,Mettre en place un certificat SSL (HTTPS) pour...
2,27.22,13,26,Définir une procédure formelle de réaction en ...
3,27.22,13,27,Prévoir une chaîne d'alerte immédiate (hiérarc...
4,17.42,13,30,Documenter une procédure de reprise post-incid...



=== Boutique Multimode (49 recommandations) ===


,score_priorite,id_domaine,id_regle,titre_mesure
0,14.35,10,17,Restreindre les comptes administrateurs aux se...
1,14.35,12,22,Chiffrer les données sensibles avant tout stoc...
2,14.35,13,26,Définir une procédure formelle de réaction en ...
3,14.35,13,27,Prévoir une chaîne d'alerte immédiate (hiérarc...
4,9.19,9,6,Installer un antivirus à jour sur tous les pos...



=== Assurance Maghreb (14 recommandations) ===


,score_priorite,id_domaine,id_regle,titre_mesure
0,26.67,12,22,Chiffrer les données sensibles avant tout stoc...
1,26.67,12,23,Mettre en place un certificat SSL (HTTPS) pour...
2,17.07,15,33,Mettre en conformité la collecte de données pe...
3,17.07,7,45,Mettre en place un programme de sensibilisatio...
4,17.07,7,47,Sensibiliser les employés à ne jamais divulgue...


### Vérifications de cohérence attendues

- **Clinique Santé Nord** et **Boutique Multimode** doivent avoir le plus de
  recommandations (maturité faible sur presque tous les domaines).
- **Assurance Maghreb** et **Startup FinTech Wave** doivent avoir peu de
  recommandations mais des scores élevés sur celles-ci (entreprises matures
  avec des lacunes ciblées à fort impact — crypto notamment).
- Aucune règle fille liée à `traite_donnees_sensibles` ne doit apparaître
  pour **Usine Textile Atlas** ou **Boutique Multimode** (données non
  sensibles).


In [11]:
# Vérification automatique des points ci-dessus
assert len(plans["Clinique Santé Nord"]) > len(plans["Assurance Maghreb"])
assert len(plans["Boutique Multimode"]) > len(plans["Startup FinTech Wave"])

ids_atlas = {r["id_regle"] for r in plans["Usine Textile Atlas"]}
ids_multimode = {r["id_regle"] for r in plans["Boutique Multimode"]}
regles_filles_sensibles = {4, 23, 32}
assert not (ids_atlas & regles_filles_sensibles)
assert not (ids_multimode & regles_filles_sensibles)

print("Toutes les vérifications de cohérence sont passées.")


Toutes les vérifications de cohérence sont passées.


## 6. Suite de tests automatisés (`pytest`)

Exécute `test_rule_engine.py` et `test_scoring.py` (53 tests) directement
depuis le notebook, pour avoir une preuve d'exécution complète en un seul
document.


In [12]:
resultat = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    cwd=BACKEND_DIR,
    capture_output=True,
    text=True,
)
print(resultat.stdout[-4000:])
if resultat.returncode != 0:
    print("STDERR:\n", resultat.stderr[-2000:])



STDERR:
 c:\Users\hp\AppData\Local\Programs\Python\Python314\python.exe: No module named pytest



## Conclusion — Sprint 3

- Le moteur d'inférence gère correctement les conditions simples, composées
  (`et` avec égalité ou appartenance) et la hiérarchie règle mère / règle
  fille (y compris sur plusieurs niveaux).
- Le corpus réel (53 règles / 53 mesures) est chargé et validé ; les 3
  couples mère/fille du guide CMRPI/AUSIM se comportent comme attendu.
- Le scoring combine gravité de la règle, impact de la mesure, contexte de
  risque (Q1-Q9) et pertinence sectorielle — méthodologie qualitative
  documentée et justifiée, conformément au retour de Pr. Ajana.
- Testé sur 6 profils PME fictifs contrastés (secteurs, tailles, budgets,
  présence de données sensibles, RGPD, historiques d'incident différents) :
  le comportement du moteur reste cohérent dans tous les cas.
- 53/53 tests automatisés passent.

**Prêt pour le Sprint 4** (pipeline RAG/LLM).
